# Tabla de promoción unificada y manifiesto frozen para paper trading

Cierre del arco experimental fusionando los **dos carriles** post-experimento ya cerrados:

- **Carril primary** (`_exp2_regen_risk_orchestrator`): 32 ramas frozen del panel greedy contextual, cada una reevaluada con EVT/CVaR + sizing sobre la señal del modelo principal (`lstm`, `residual_hybrid`, `xgboost`, `voting`).
- **Carril metalabeling** (`<study>/execution_risk/post_experiment_metalabeling/frozen_20260520T0238Z/`): 30 ramas del greedy *meta-labeling* contextual con EVT/CVaR + sizing aplicados sobre la **señal meta-filtrada** (decisión del primario tras el veto del meta-modelo).

La promoción colapsa ambos carriles en **una configuración campeona por símbolo** y produce el manifiesto que el runtime de paper trading consume.

## Criterio de promoción (idéntico para ambos carriles)

1. La celda debe haber pasado el gate de aceptación de execution risk (`is_accepted = True`).
2. El brazo `con_filtro` (EVT/CVaR) debe mantener `strategy_dsr` >= 0.95 (umbral del cierre confirmatorio).
3. La familia ganadora del greedy no puede ser una familia trivial (excluimos `no_skill`).
4. Entre las celdas elegibles de un símbolo, ranking por `strategy_sharpe` de la política de sizing comparada (criterio primario), con `strategy_sortino`, `strategy_total_return` y `filtered_arm_strategy_dsr` como desempates.
5. La política de sizing se evalúa sobre la salida del brazo filtrado del execution risk (modo `reuse_frozen_policy_outputs` exigido por el contrato `paper_trading_position_sizing_alignment`).

## Alcance del pool elegible

La promoción **no** está restringida al nodo `winner_subset` del greedy contextual. Rankea **todas** las celdas elegibles ingeridas desde los reports congelados del orchestrator post-riesgo (carril 2.5: EVT/CVaR + sizing): cualquier rama explorada en el árbol que haya pasado `is_accepted` y el umbral de DSR entra en competición por símbolo.

Por eso un activo puede promoverse con un bloque contextual (`sentiment`, `derivatives_futures`, etc.) aunque el veredicto greedy de esa rama fuera `core_only`: gana la celda con mejor `sizing_strategy_sharpe` entre las elegibles, no necesariamente el nodo marcado como ganador del árbol. Las métricas de referencia son las del carril 2.5 (reports `post_experiment_risk_lane_report.json` / `position_sizing_summary.json`), no el ranking OOS del greedy alone.

El campeón Exp1 por `(symbol, method)` tampoco es criterio directo de promoción: aquí se elige **una** configuración por símbolo cruzando carriles, métodos de etiquetado y políticas de sizing.

La regla y los carriles ingeridos quedan firmados en el manifiesto bajo `promotion_rule` y `lanes_ingested`.

Los insumos provienen de los carriles 2.5 cerrados en `05_modelado_predictivo/06_resumen_carril_riesgo_greedy_contextual.ipynb` (primary) y `05_modelado_predictivo/08_resumen_carril_riesgo_meta_labeling_contextual.ipynb` (*meta-labeling*).

## Productos

Salidas bajo `reports/validation/promotion/<stamp>/`:

- `promotion_table.csv` — filas (celda × política de sizing) con métricas leídas de los reports congelados.
- `promotion_table_champions.csv` — una fila por símbolo con el ganador unificado.
- `promotion_table_lane_comparison.csv` — mejor primary vs. mejor *meta-labeling* por símbolo.
- `promotion_manifest.json` — contrato firmado consumido por el runtime de *paper trading* y por `02_comprobacion_sanidad_senales.ipynb`.

In [2]:
from __future__ import annotations

import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.options.display.max_columns = 60
pd.options.display.max_rows = 200

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'src' / 'validation' / 'promotion.py').exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'src' / 'validation' / 'promotion.py').exists():
    raise SystemExit('No se encuentra el repo TFG (src/validation/promotion.py).')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.validation.promotion import (
    DEFAULT_METALABELING_STAMP,
    DEFAULT_METALABELING_SUBDIR_NAME,
    DEFAULT_PROMOTION_RULE,
    LANE_METALABELING,
    LANE_PRIMARY,
    PromotionRule,
    build_promotion_manifest,
    build_promotion_table,
    describe_promotion_contract,
    diagnose_orchestrator_cell_loading,
    discover_cells_from_metalabeling_lane,
    discover_cells_from_orchestrator,
    parse_active_blocks,
    select_champion_per_symbol,
    select_eligible_rows,
    write_promotion_manifest,
)

COLUMNAS_ES = {
    'study_id': 'id_estudio',
    'lane': 'carril',
    'primary_family': 'familia_primaria',
    'symbol': 'simbolo',
    'labeling_method': 'metodo_etiquetado',
    'family': 'familia',
    'injection_strategy': 'estrategia_inyeccion',
    'active_blocks': 'bloques_activos',
    'branch_stamp': 'sello_rama',
    'frozen_stamp': 'sello_frozen',
    'active_source_blocks': 'bloques_fuente_activos',
    'risk_policy_name': 'politica_riesgo',
    'risk_arm_used_for_sizing': 'brazo_riesgo_para_sizing',
    'control_arm_strategy_dsr': 'dsr_brazo_control',
    'control_arm_strategy_sharpe': 'sharpe_brazo_control',
    'filtered_arm_strategy_dsr': 'dsr_brazo_filtrado',
    'filtered_arm_strategy_sharpe': 'sharpe_brazo_filtrado',
    'filtered_arm_realized_cvar': 'cvar_realizado_filtrado',
    'filtered_arm_blocked_trade_rate': 'tasa_bloqueo_filtrado',
    'filtered_arm_coverage': 'cobertura_filtrada',
    'acceptance_gate_passed': 'gate_aceptacion_ok',
    'sizing_policy_name': 'politica_sizing',
    'sizing_strategy_sharpe': 'sharpe_estrategia_sizing',
    'sizing_strategy_sortino': 'sortino_estrategia_sizing',
    'sizing_strategy_total_return': 'retorno_total_sizing',
    'sizing_strategy_max_drawdown': 'maxdd_sizing',
    'sizing_hit_rate': 'hit_rate_sizing',
    'sizing_turnover': 'turnover_sizing',
    'sizing_n_obs': 'n_obs_sizing',
    'risk_report_path': 'ruta_informe_riesgo',
    'sizing_summary_path': 'ruta_resumen_sizing',
    'n_cells': 'n_celdas',
    'n_rows': 'n_filas',
    'needs_contextual': 'requiere_contextual',
    'delta_sharpe_meta_minus_primary': 'delta_sharpe_meta_menos_primary',
    'unified_champion_lane': 'carril_campeon_unificado',
    'primary_lane': 'primary_carril',
    'primary_primary_family': 'primary_familia_primaria',
    'primary_labeling_method': 'primary_metodo_etiquetado',
    'primary_active_blocks': 'primary_bloques_activos',
    'primary_sizing_policy_name': 'primary_politica_sizing',
    'primary_sizing_strategy_sharpe': 'primary_sharpe_sizing',
    'primary_filtered_arm_strategy_dsr': 'primary_dsr_filtrado',
    'meta_lane': 'meta_carril',
    'meta_family': 'meta_familia',
    'meta_primary_family': 'meta_familia_primaria',
    'meta_labeling_method': 'meta_metodo_etiquetado',
    'meta_active_blocks': 'meta_bloques_activos',
    'meta_sizing_policy_name': 'meta_politica_sizing',
    'meta_sizing_strategy_sharpe': 'meta_sharpe_sizing',
    'meta_filtered_arm_strategy_dsr': 'meta_dsr_filtrado',
}

REGLA_ES = {
    'require_acceptance_gate': 'exigir_gate_aceptacion',
    'min_filtered_arm_dsr': 'dsr_minimo_brazo_filtrado',
    'require_filtered_arm_admissible': 'exigir_brazo_filtrado_admisible',
    'primary_metric': 'metrica_primaria',
    'tie_breakers': 'desempates',
    'excluded_families': 'familias_excluidas',
}


def a_es(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [COLUMNAS_ES.get(col, col) for col in out.columns]
    return out


def display_es(df: pd.DataFrame) -> None:
    display(a_es(df))


METRICA_ES = {
    'sizing_strategy_sharpe': 'sharpe_estrategia_sizing',
    'sizing_strategy_sortino': 'sortino_estrategia_sizing',
    'sizing_strategy_total_return': 'retorno_total_sizing',
    'filtered_arm_strategy_dsr': 'dsr_brazo_filtrado',
}


def regla_es(rule_payload: dict) -> pd.Series:
    out: dict[str, object] = {}
    for key, value in rule_payload.items():
        label = REGLA_ES.get(key, key)
        if key == 'primary_metric' and isinstance(value, str):
            out[label] = METRICA_ES.get(value, value)
        elif key == 'tie_breakers' and isinstance(value, (list, tuple)):
            out[label] = [METRICA_ES.get(v, v) for v in value]
        else:
            out[label] = value
    return pd.Series(out)


## 1. Localizar artefactos de los dos carriles

- Carril **primary**: orchestrator_summary del rerun firmado.
- Carril **metalabeling**: el bundle bajo `<study>/execution_risk/post_experiment_metalabeling/<frozen_stamp>/<family>/` (sello por defecto `frozen_20260520T0238Z`).

In [3]:
EXPERIMENTS_ROOT = REPO_ROOT / 'reports' / 'validation' / 'experiments'
ORCHESTRATOR_DIR = EXPERIMENTS_ROOT / '_exp2_regen_risk_orchestrator'
GREEDY_SUMMARY = EXPERIMENTS_ROOT / 'contextual_greedy' / '20260505T012242Z' / 'greedy_summary.json'
META_GREEDY_SUMMARY = EXPERIMENTS_ROOT / 'contextual_metalabeling' / '20260513T182131Z' / 'metalabeling_greedy_summary.json'
META_FROZEN_STAMP = DEFAULT_METALABELING_STAMP

candidates = sorted(p for p in ORCHESTRATOR_DIR.glob('*/orchestrator_summary.json'))
if not candidates:
    raise SystemExit(f'No hay orchestrator_summary.json bajo {ORCHESTRATOR_DIR}')
ORCH_SUMMARY = candidates[-1]

print('Resumen orchestrator (carril primary)     :', ORCH_SUMMARY.relative_to(REPO_ROOT))
print('Resumen greedy (carril primary)           :', GREEDY_SUMMARY.relative_to(REPO_ROOT))
print('Resumen greedy (carril metalabeling)      :', META_GREEDY_SUMMARY.relative_to(REPO_ROOT))
print('Sello frozen metalabeling                 :', META_FROZEN_STAMP)

Resumen orchestrator (carril primary)     : reports/validation/experiments/_exp2_regen_risk_orchestrator/20260517T223949Z/orchestrator_summary.json
Resumen greedy (carril primary)           : reports/validation/experiments/contextual_greedy/20260505T012242Z/greedy_summary.json
Resumen greedy (carril metalabeling)      : reports/validation/experiments/contextual_metalabeling/20260513T182131Z/metalabeling_greedy_summary.json
Sello frozen metalabeling                 : frozen_20260520T0238Z


## 2. Cargar celdas de los dos carriles y construir la tabla unificada

Cada celda produce una fila por política de sizing (`fixed_control`, `vol_target`, `identity` cuando aplica). El carril metalabeling segrega por `primary_family`.

In [4]:
primary_snapshots = discover_cells_from_orchestrator(ORCH_SUMMARY, repo_root=REPO_ROOT)
meta_snapshots = discover_cells_from_metalabeling_lane(EXPERIMENTS_ROOT, frozen_stamp=META_FROZEN_STAMP)
print(f'Celdas carril primary                    : {len(primary_snapshots)}')
print(f'Celdas carril metalabeling               : {len(meta_snapshots)}')

if not primary_snapshots:
    diag = diagnose_orchestrator_cell_loading(ORCH_SUMMARY, repo_root=REPO_ROOT)
    print('AVISO: carril primary vacío. Diagnóstico orchestrator ->')
    print(
        '  entradas={orchestrator_entries}, risk_report_encontrados={risk_report_found}, '
        'sizing_encontrados={sizing_summary_found}, celdas_cargables={cells_loadable}'.format(
            **diag
        )
    )
    print(
        '  Causa habitual: rutas absolutas de otro host/OS en orchestrator_summary.json; '
        'deben remapearse bajo REPO_ROOT via reports/...'
    )

rows = build_promotion_table(primary_snapshots) + build_promotion_table(meta_snapshots)
table = pd.DataFrame(rows)
table['active_source_blocks'] = table['active_blocks'].map(lambda x: list(parse_active_blocks(str(x))))

print(f'Filas totales                            : {len(table)}')
print('  por carril:')
lane_counts = table.groupby('lane').size().rename('conteo')
lane_counts.index = lane_counts.index.map({'primary': 'primary', 'metalabeling': 'metalabeling'})
print(lane_counts.to_string())
display_es(table.head())

Celdas carril primary                    : 32
Celdas carril metalabeling               : 30
Filas totales                            : 154
  por carril:
lane
metalabeling    90
primary         64


,id_estudio,carril,familia_primaria,simbolo,metodo_etiquetado,familia,estrategia_inyeccion,bloques_activos,sello_rama,sello_frozen,politica_riesgo,brazo_riesgo_para_sizing,dsr_brazo_control,sharpe_brazo_control,dsr_brazo_filtrado,sharpe_brazo_filtrado,cvar_realizado_filtrado,tasa_bloqueo_filtrado,cobertura_filtrada,gate_aceptacion_ok,politica_sizing,sharpe_estrategia_sizing,sortino_estrategia_sizing,retorno_total_sizing,maxdd_sizing,hit_rate_sizing,turnover_sizing,n_obs_sizing,ruta_informe_riesgo,ruta_resumen_sizing,bloques_fuente_activos
0,greedy_bnbusdt_triple_barrier_residual_hybrid_...,primary,,BNBUSDT,triple_barrier,residual_hybrid,staged,core_only,20260505T012242Z,20260517T134241Z,evt_cvar,con_filtro,0.820840,0.026892,0.998874,0.025655,-0.055252,0.018947,0.793860,True,fixed_control,0.025655,0.038555,11.288783,-0.974842,0.406082,0.422715,17100.0,/app/reports/validation/experiments/greedy_bnb...,/app/reports/validation/experiments/greedy_bnb...,[core_only]
1,greedy_bnbusdt_triple_barrier_residual_hybrid_...,primary,,BNBUSDT,triple_barrier,residual_hybrid,staged,core_only,20260505T012242Z,20260517T134241Z,evt_cvar,con_filtro,0.820840,0.026892,0.998874,0.025655,-0.055252,0.018947,0.793860,True,vol_target,0.029506,0.061404,9.618460,-0.938367,0.406082,0.449514,17100.0,/app/reports/validation/experiments/greedy_bnb...,/app/reports/validation/experiments/greedy_bnb...,[core_only]
2,greedy_bnbusdt_triple_barrier_residual_hybrid_...,primary,,BNBUSDT,triple_barrier,residual_hybrid,monolithic,onchain,20260505T012242Z,20260517T134241Z,evt_cvar,con_filtro,0.051219,0.007349,0.712358,0.006268,-0.052623,0.015263,0.715556,False,fixed_control,0.006268,0.009301,2.594368,-0.998702,0.361637,0.326218,17100.0,/app/reports/validation/experiments/greedy_bnb...,/app/reports/validation/experiments/greedy_bnb...,[onchain]
3,greedy_bnbusdt_triple_barrier_residual_hybrid_...,primary,,BNBUSDT,triple_barrier,residual_hybrid,monolithic,onchain,20260505T012242Z,20260517T134241Z,evt_cvar,con_filtro,0.051219,0.007349,0.712358,0.006268,-0.052623,0.015263,0.715556,False,vol_target,0.008224,0.016964,2.581822,-0.989031,0.361637,0.341033,17100.0,/app/reports/validation/experiments/greedy_bnb...,/app/reports/validation/experiments/greedy_bnb...,[onchain]
4,greedy_btcusdt_fixed_horizon_residual_hybrid_s...,primary,,BTCUSDT,fixed_horizon,residual_hybrid,staged,microstructure,20260505T012242Z,20260517T134241Z,evt_cvar,con_filtro,0.806451,0.026123,0.999729,0.026123,-0.075442,0.000000,0.885682,True,fixed_control,0.026123,0.034416,14.671602,-0.997790,0.454410,0.582508,17460.0,/app/reports/validation/experiments/greedy_btc...,/app/reports/validation/experiments/greedy_btc...,[microstructure]


## 3. Cobertura por símbolo y por método de etiquetado

Diagnóstico previo a la promoción para detectar símbolos sin candidatos elegibles.

Si solo aparece el carril `metalabeling`, el carril **primary** no cargó artefactos (revisar el diagnóstico de la sección 2 y que `orchestrator_summary.json` remapee bajo `REPO_ROOT`).

In [5]:
coverage = (
    table.groupby(['lane', 'symbol', 'labeling_method'])
    .agg(n_cells=('study_id', 'nunique'), n_rows=('study_id', 'size'))
    .reset_index()
    .sort_values(['symbol', 'lane', 'labeling_method'])
)
display_es(coverage)

,carril,simbolo,metodo_etiquetado,n_celdas,n_filas
0,metalabeling,BNBUSDT,fixed_horizon,2,6
1,metalabeling,BNBUSDT,trend_scanning,2,6
2,metalabeling,BNBUSDT,triple_barrier,1,6
15,primary,BNBUSDT,fixed_horizon,2,4
16,primary,BNBUSDT,trend_scanning,2,4
17,primary,BNBUSDT,triple_barrier,2,4
3,metalabeling,BTCUSDT,fixed_horizon,2,6
4,metalabeling,BTCUSDT,trend_scanning,1,6
5,metalabeling,BTCUSDT,triple_barrier,2,6
18,primary,BTCUSDT,fixed_horizon,2,4


## 4. Aplicar el criterio de promoción

Documentamos la regla efectiva (firmada en el manifiesto) y filtramos las filas elegibles.

Campos de la regla (etiquetas en español en la salida):

- `exigir_gate_aceptacion` -> la celda debe haber pasado el gate de execution risk.
- `dsr_minimo_brazo_filtrado` -> umbral mínimo de DSR del brazo filtrado EVT/CVaR.
- `exigir_brazo_filtrado_admisible` -> el brazo filtrado debe ser admisible según el informe de riesgo.
- `metrica_primaria` -> métrica principal del ranking (`sharpe_estrategia_sizing`).
- `desempates` -> orden secundario: sortino, retorno total y DSR filtrado.
- `familias_excluidas` -> familias descartadas (p. ej. `no_skill`).

In [6]:
rule = PromotionRule(
    require_acceptance_gate=True,
    min_filtered_arm_dsr=0.95,
    require_filtered_arm_admissible=True,
    primary_metric='sizing_strategy_sharpe',
    tie_breakers=('sizing_strategy_sortino', 'sizing_strategy_total_return', 'filtered_arm_strategy_dsr'),
    excluded_families=('no_skill',),
)
display(regla_es(rule.to_payload()))

exigir_gate_aceptacion                                                          True
dsr_minimo_brazo_filtrado                                                       0.95
exigir_brazo_filtrado_admisible                                                 True
metrica_primaria                                            sharpe_estrategia_sizing
desempates                         [sortino_estrategia_sizing, retorno_total_sizi...
familias_excluidas                                                        [no_skill]
dtype: object

In [7]:
eligible = pd.DataFrame(select_eligible_rows(rows, rule=rule))
if eligible.empty:
    print('Sin candidatos tras aplicar la regla; revisar gates/DSR.')
else:
    print(f'Filas elegibles totales                 : {len(eligible)}')
    print('  por carril:')
    print(eligible.groupby('lane').size().to_string())
    display_cols = [
        'lane', 'primary_family', 'symbol', 'labeling_method', 'family', 'active_blocks',
        'sizing_policy_name', 'sizing_strategy_sharpe', 'sizing_strategy_sortino',
        'sizing_strategy_total_return', 'filtered_arm_strategy_dsr',
        'filtered_arm_blocked_trade_rate', 'filtered_arm_coverage',
    ]
    eligible_view = eligible[display_cols].sort_values(
        ['symbol', 'sizing_strategy_sharpe'],
        ascending=[True, False],
    )
    display_es(eligible_view.head(40))

Filas elegibles totales                 : 37
  por carril:
lane
metalabeling     9
primary         28


,carril,familia_primaria,simbolo,metodo_etiquetado,familia,bloques_activos,politica_sizing,sharpe_estrategia_sizing,sortino_estrategia_sizing,retorno_total_sizing,dsr_brazo_filtrado,tasa_bloqueo_filtrado,cobertura_filtrada
20,primary,,BNBUSDT,trend_scanning,residual_hybrid,core_only,fixed_control,0.074153,0.131230,115.163600,1.000000,0.052632,0.947368
21,primary,,BNBUSDT,trend_scanning,residual_hybrid,core_only,vol_target,0.065458,0.116237,29.299683,1.000000,0.052632,0.947368
22,primary,,BNBUSDT,trend_scanning,residual_hybrid,onchain,fixed_control,0.060574,0.098555,94.161294,1.000000,0.052632,0.947368
8,primary,,BNBUSDT,fixed_horizon,residual_hybrid,onchain,fixed_control,0.054997,0.080710,41.552903,1.000000,0.000000,0.980585
10,primary,,BNBUSDT,fixed_horizon,residual_hybrid,core_only,fixed_control,0.054215,0.079181,41.028654,1.000000,0.000000,0.983684
23,primary,,BNBUSDT,trend_scanning,residual_hybrid,onchain,vol_target,0.053467,0.092538,23.949265,1.000000,0.052632,0.947368
11,primary,,BNBUSDT,fixed_horizon,residual_hybrid,core_only,vol_target,0.047168,0.071338,18.166861,1.000000,0.000000,0.983684
9,primary,,BNBUSDT,fixed_horizon,residual_hybrid,onchain,vol_target,0.042776,0.064662,16.404092,1.000000,0.000000,0.980585
1,primary,,BNBUSDT,triple_barrier,residual_hybrid,core_only,vol_target,0.029506,0.061404,9.618460,0.998874,0.018947,0.793860
0,primary,,BNBUSDT,triple_barrier,residual_hybrid,core_only,fixed_control,0.025655,0.038555,11.288783,0.998874,0.018947,0.793860


## 5. Selección de campeón por símbolo

Dos vistas complementarias:

1. **Campeón unificado:** una fila por activo con métricas completas y columna `requiere_contextual` (derivada de `bloques_fuente_activos`; indica si el paper trading debe ingestar fuentes externas además de OHLCV).
2. **Comparativa por carril:** mejor candidato elegible del carril primary vs. metalabeling por símbolo, con el delta de Sharpe y el carril ganador. La tabla 1 solo muestra el ganador; esta explica *por qué* ganó ese carril (p. ej. XRP, donde meta supera a primary).

In [8]:
champions = select_champion_per_symbol(rows, rule=rule)
promotion_df = pd.DataFrame(list(champions.values())).sort_values('symbol').reset_index(drop=True)
promotion_df['active_source_blocks'] = promotion_df['active_blocks'].map(
    lambda x: list(parse_active_blocks(str(x)))
)
promotion_df['needs_contextual'] = promotion_df['active_source_blocks'].map(
    lambda blocks: any(b for b in blocks if b != 'core_only')
)
summary_cols = [
    'symbol', 'lane', 'primary_family', 'labeling_method', 'family',
    'active_blocks', 'active_source_blocks', 'needs_contextual',
    'sizing_policy_name', 'sizing_strategy_sharpe', 'sizing_strategy_sortino',
    'sizing_strategy_total_return', 'sizing_strategy_max_drawdown',
    'filtered_arm_strategy_dsr', 'filtered_arm_blocked_trade_rate',
    'filtered_arm_coverage', 'risk_policy_name', 'study_id',
]
display_es(promotion_df[summary_cols])

,simbolo,carril,familia_primaria,metodo_etiquetado,familia,bloques_activos,bloques_fuente_activos,requiere_contextual,politica_sizing,sharpe_estrategia_sizing,sortino_estrategia_sizing,retorno_total_sizing,maxdd_sizing,dsr_brazo_filtrado,tasa_bloqueo_filtrado,cobertura_filtrada,politica_riesgo,id_estudio
0,BNBUSDT,primary,,trend_scanning,residual_hybrid,core_only,[core_only],False,fixed_control,0.074153,0.131230,115.163600,-1.000000,1.000000,0.052632,0.947368,evt_cvar,greedy_bnbusdt_trend_scanning_residual_hybrid_...
1,BTCUSDT,primary,,trend_scanning,lstm,core_only,[core_only],False,vol_target,0.074472,0.132691,34.059134,-0.999160,1.000000,0.020160,0.976460,evt_cvar,greedy_btcusdt_trend_scanning_lstm_none_core_o...
2,ETHUSDT,primary,,fixed_horizon,residual_hybrid,sentiment,[sentiment],True,fixed_control,0.027719,0.037597,20.609661,-0.999952,0.999876,0.000000,0.961340,evt_cvar,greedy_ethusdt_fixed_horizon_residual_hybrid_s...
3,SOLUSDT,primary,,fixed_horizon,residual_hybrid,derivatives_futures,[derivatives_futures],True,fixed_control,0.036540,0.053055,23.461094,-1.000000,0.999939,0.000000,0.998452,evt_cvar,greedy_solusdt_fixed_horizon_residual_hybrid_s...
4,XRPUSDT,metalabeling,lstm,trend_scanning,meta_labeling,core_only,[core_only],False,vol_target,0.054789,0.064303,16.660739,-0.986977,0.999999,0.000000,0.477091,evt_cvar,supervised_trend_scanning_4h_v1_core_only-fc86...


**Comparativa por símbolo (mejor primary vs mejor metalabeling).** Solo se muestra cuando existen candidatos elegibles en ambos carriles; si un carril no tiene filas elegibles para ese símbolo, las columnas meta quedan vacías (ETH, SOL).

In [9]:
def _best_per_lane(eligible_df: pd.DataFrame, lane: str) -> pd.DataFrame:
    sub = eligible_df[eligible_df['lane'] == lane]
    if sub.empty:
        return sub
    return (
        sub.sort_values(['symbol', 'sizing_strategy_sharpe'], ascending=[True, False])
        .groupby('symbol', as_index=False)
        .first()
    )


best_primary = _best_per_lane(eligible, LANE_PRIMARY)
best_meta = _best_per_lane(eligible, LANE_METALABELING)

cmp_cols = [
    'symbol', 'lane', 'family', 'primary_family', 'labeling_method',
    'active_blocks', 'sizing_policy_name',
    'sizing_strategy_sharpe', 'filtered_arm_strategy_dsr',
]
left = best_primary[cmp_cols].rename(
    columns={c: f'primary_{c}' for c in cmp_cols if c != 'symbol'}
)
right = best_meta[cmp_cols].rename(
    columns={c: f'meta_{c}' for c in cmp_cols if c != 'symbol'}
)
comparison = (
    left.merge(right, on='symbol', how='outer')
    .sort_values('symbol')
    .reset_index(drop=True)
)
comparison['delta_sharpe_meta_minus_primary'] = (
    comparison['meta_sizing_strategy_sharpe'] - comparison['primary_sizing_strategy_sharpe']
)
comparison['unified_champion_lane'] = comparison['symbol'].map(
    lambda s: champions[s]['lane'] if s in champions else None
)
display_es(comparison)

,simbolo,primary_carril,familia_primaria,primary_familia_primaria,primary_metodo_etiquetado,primary_bloques_activos,primary_politica_sizing,primary_sharpe_sizing,primary_dsr_filtrado,meta_carril,meta_familia,meta_familia_primaria,meta_metodo_etiquetado,meta_bloques_activos,meta_politica_sizing,meta_sharpe_sizing,meta_dsr_filtrado,delta_sharpe_meta_menos_primary,carril_campeon_unificado
0,BNBUSDT,primary,residual_hybrid,,trend_scanning,core_only,fixed_control,0.074153,1.000000,metalabeling,meta_labeling,lstm,trend_scanning,core_only,identity,0.020968,0.997391,-0.053184,primary
1,BTCUSDT,primary,lstm,,trend_scanning,core_only,vol_target,0.074472,1.000000,metalabeling,meta_labeling,lstm,trend_scanning,core_only,identity,0.022903,0.998378,-0.051569,primary
2,ETHUSDT,primary,residual_hybrid,,fixed_horizon,sentiment,fixed_control,0.027719,0.999876,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,primary
3,SOLUSDT,primary,residual_hybrid,,fixed_horizon,derivatives_futures,fixed_control,0.036540,0.999939,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,primary
4,XRPUSDT,primary,residual_hybrid,,trend_scanning,sentiment,vol_target,0.035155,0.999936,metalabeling,meta_labeling,lstm,trend_scanning,core_only,vol_target,0.054789,0.999999,0.019634,metalabeling


## 6. Persistir tabla, manifiesto frozen y evidencia

Salidas auditables bajo `reports/validation/promotion/<stamp>/` que el paper driver consume:

- `promotion_table.csv`: filas (celda × política de sizing) con todas las métricas leídas de los reports congelados, etiquetadas por carril (`primary` / `metalabeling`).
- `promotion_table_champions.csv`: una fila por símbolo con el ganador unificado, su carril y las fuentes que el modelo necesita.
- `promotion_table_lane_comparison.csv`: para cada símbolo, mejor candidato primary y mejor candidato metalabeling lado a lado, con `delta_sharpe_meta_minus_primary` y el carril ganador final.
- `promotion_manifest.json`: contrato firmado (`manifest_fingerprint_sha256`) que el runtime de *paper trading* lee y reproduce sin recalibrar. Contiene la regla aplicada, para cada símbolo su `lane`, `primary_family`, `study_id`, familia ganadora, fuentes activas, política de sizing/riesgo, métricas DSR/Sharpe y rutas a artefactos congelados, más `lanes_ingested` para auditoría.

In [10]:
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT_DIR = REPO_ROOT / 'reports' / 'validation' / 'promotion' / stamp
OUT_DIR.mkdir(parents=True, exist_ok=True)

table_path = OUT_DIR / 'promotion_table.csv'
table.to_csv(table_path, index=False)
promotion_full_path = OUT_DIR / 'promotion_table_champions.csv'
promotion_df.to_csv(promotion_full_path, index=False)
comparison_path = OUT_DIR / 'promotion_table_lane_comparison.csv'
comparison.to_csv(comparison_path, index=False)

manifest = build_promotion_manifest(
    champions,
    rule=rule,
    orchestrator_summary_path=ORCH_SUMMARY,
    greedy_summary_path=GREEDY_SUMMARY,
    extra_metadata={
        'notebook': 'notebooks/06_paper_trading/01_tabla_promocion.ipynb',
        'contract_describe': describe_promotion_contract(),
        'lanes_ingested': {
            'primary': {
                'orchestrator_summary': ORCH_SUMMARY.as_posix(),
                'greedy_summary': GREEDY_SUMMARY.as_posix(),
                'cells_loaded': len(primary_snapshots),
            },
            'metalabeling': {
                'experiments_root': EXPERIMENTS_ROOT.as_posix(),
                'frozen_stamp': META_FROZEN_STAMP,
                'subdir_name': DEFAULT_METALABELING_SUBDIR_NAME,
                'greedy_summary': META_GREEDY_SUMMARY.as_posix(),
                'cells_loaded': len(meta_snapshots),
            },
        },
        'champions_lane_breakdown': (
            promotion_df.groupby('lane')['symbol'].count().to_dict()
        ),
    },
)
manifest_path = write_promotion_manifest(manifest, OUT_DIR / 'promotion_manifest.json')

{
    'output_dir': OUT_DIR.as_posix(),
    'manifest': manifest_path.as_posix(),
    'comparison_csv': comparison_path.as_posix(),
}

{'output_dir': '/app/reports/validation/promotion/20260604T130713Z',
 'manifest': '/app/reports/validation/promotion/20260604T130713Z/promotion_manifest.json',
 'comparison_csv': '/app/reports/validation/promotion/20260604T130713Z/promotion_table_lane_comparison.csv'}

## 7. Conclusiones

- Se han ingerido **ambos carriles** (primary: **32** celdas del orchestrator post-riesgo; metalabeling: **30** celdas del bundle frozen) y construido una tabla unificada de **154** filas.
- Tras aplicar el criterio de promoción (gate de aceptación, `DSR >= 0.95`, exclusión de `no_skill`), quedan **37** filas elegibles (**28** primary + **9** metalabeling).
- La selección de campeón por símbolo produce **5** configuraciones: **4** del carril primary (BNB, BTC, ETH, SOL) y **1** del carril metalabeling (XRP), coherente con el manifiesto operativo `20260604T130713Z`.
- ETH y SOL promovieron con bloques contextuales (`sentiment` y `derivatives_futures` respectivamente), lo que exige que el pipeline de *paper trading* ingeste esas fuentes externas además de OHLCV.
- XRP es el único símbolo donde metalabeling supera a primary en Sharpe post-sizing, evidenciando que el filtro del meta-modelo aporta valor en ese activo.
- El manifiesto frozen firmado (`promotion_manifest.json`) queda listo para consumo por el runtime de *paper trading*; el siguiente paso operativo es `02_comprobacion_sanidad_senales.ipynb`.